> ⚠️ **EXCLU depuis le 2026-11-12** : la règle « 100 % local » interdit le calcul dans le cloud. Conservé pour l'historique ; utiliser un PC personnel (voir `master01/SELECTION_MODELES_LOCAUX.md`).

# PERSONA IA — MASTER 01 · Phase B/D/E/F sur GPU (Colab T4)

**Règles** : pas d'interface Gradio publique, pas de tunnel, pas de photo personnelle dans le dépôt.
Le mode démonstration utilise une **personne fictive** générée. Le mode réel lit des références
**autorisées** depuis un dossier privé que vous montez vous-même.

Pourquoi un venv Python 3.11 : sur Colab Python 3.13, `torch==2.3.1` (requis par la pile PhotoMaker V2
verrouillée) n'a pas de roue cp313 → c'est la cause des échecs d'import `diffusers` observés.
On ne met plus à jour `diffusers` à la main : tout vient de `requirements-gpu.lock`.

Exécution : *Exécution → Modifier le type d'exécution → GPU T4*, puis lancer les cellules dans l'ordre.

In [ ]:
!nvidia-smi
!python3 --version
!df -h /content | tail -1

## 1. Sources PERSONA IA (branche de travail) et PhotoMaker (commit enregistré)

In [ ]:
%cd /content
# Dépôt privé : Colab demandera un jeton GitHub en lecture seule si nécessaire (ne jamais l'écrire dans le notebook).
!git clone -q -b ccr-8f135e00-hwmdvs https://github.com/bubth720/persona-ia.git || (cd persona-ia && git pull -q)
!git clone -q https://github.com/TencentARC/PhotoMaker.git || true
!git -C PhotoMaker rev-parse HEAD | tee /content/photomaker_commit.txt
!git -C persona-ia rev-parse HEAD | tee /content/persona_commit.txt

## 2. Environnement propre et verrouillé (Python 3.11 via uv)

In [ ]:
!pip -q install uv
!uv venv -q --python 3.11 /content/venv311
!VIRTUAL_ENV=/content/venv311 uv pip install -q -r /content/persona-ia/master01/engine/requirements-gpu.lock
!/content/venv311/bin/python -c "import torch,diffusers,transformers;print(torch.__version__,torch.version.cuda,torch.cuda.is_available(),diffusers.__version__,transformers.__version__)"

## 3. Capture d'environnement + test d'import (B1)

In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/PhotoMaker:/content/persona-ia/master01/engine'
os.environ['PHOTOMAKER_COMMIT'] = open('/content/photomaker_commit.txt').read().strip()
PY = '/content/venv311/bin/python'
!mkdir -p /content/persona_out
!{PY} -m persona_engine.env_capture /content/persona_out/env_capture.json /content/PhotoMaker
!{PY} -c "from photomaker import PhotoMakerStableDiffusionXLPipeline, FaceAnalysis2, analyze_faces; print('IMPORT PhotoMaker OK')"
!{PY} -c "from photomaker import PhotoMakerStableDiffusionXLControlNetPipeline; print('ControlNet pipeline OK')" || echo "ControlNet PhotoMaker absent -> planche corps sans contrôle de pose (signalé)"

## 4. Test de chargement des poids (téléchargement HuggingFace ; vérifiez les licences dans MOTEUR_GPU.md)

In [ ]:
!{PY} - <<'PY'
import time, torch
from persona_engine.backends.photomaker_v2 import PhotoMakerV2Backend
t=time.time(); b=PhotoMakerV2Backend(); b._load_pipe(False)
print('poids chargés en', round(time.time()-t), 's ; VRAM pic', round(torch.cuda.max_memory_allocated()/2**20), 'Mo')
import hashlib; print('sha256 photomaker-v2.bin', hashlib.sha256(open(b._ckpt,'rb').read()).hexdigest())
PY

## 5. Gate B — génération réelle en mode démonstration (personne fictive)
Produit 3 références fictives, puis la planche faciale (6 vues) et corporelle (4 vues). Durée T4 estimée : 30–60 min.

In [ ]:
!{PY} -m persona_engine.run_boards --demo-synthetic --subject woman --out /content/persona_out/demo 2>&1 | tee /content/persona_out/demo_log.txt

In [ ]:
import glob, json
from IPython.display import Image, display
for p in sorted(glob.glob('/content/persona_out/demo/worker/jobs/*/board/planche_*.png')):
    print(p); display(Image(p, width=900))
print(open('/content/persona_out/demo/run_summary.json').read())

## 6. Mode réel — références autorisées (optionnel, après vérification des règles Colab)
1. Déposez les photos de la personne adulte consentante dans un dossier **privé** de votre Drive.
2. Renseignez la déclaration de consentement. Rien n'est poussé sur GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REFS = '/content/drive/MyDrive/PERSONA_REFERENCES_PRIVEES'   # à adapter
CONSENT = ''   # ex. "Usage de mes photographies et de ma ressemblance autorisé par moi-même, 2026-11-12"
assert CONSENT.strip(), 'Déclaration de consentement requise'
!{PY} -m persona_engine.run_boards --refs "{REFS}" --consent "{CONSENT}" --subject woman --out /content/persona_out/real 2>&1 | tee /content/persona_out/real_log.txt

## 7. Revue humaine (Gate D/E) et sauvegarde privée
Comparez chaque vue aux références (yeux, nez, bouche, implantation des cheveux, asymétries, mains, pieds).
Consignez la décision dans `REVUE_HUMAINE.md` (modèle dans le dépôt). Une vue ratée se régénère seule
(`JobManager.regenerate`). Puis archivez **dans votre Drive privé** :

In [ ]:
!cd /content && zip -qr persona_out.zip persona_out -x '*/refs/*' '*/refbank/originals/*' && ls -la persona_out.zip
# !cp /content/persona_out.zip /content/drive/MyDrive/   # export volontaire uniquement